calculating daily Q for all sites

In [1]:
#imports
import pandas as pd
import datetime as dt
import matplotlib.pyplot as plt
import numpy as np
import os

In [ ]:
# inputs
working_directory=r"C:\Users\duffshan\Box\Shannon_Duffy\Writing\Code For Paper Publication"
# filepaths
olddatafile=os.path.join(working_directory, "Input Data","Streamflow","Q_all_sites_1950_2024.csv")
newdatafile=os.path.join(working_directory, "Input Data","Streamflow","Q_all_sites_provisional_2015_2025.csv")
lookout_new_data_file =os.path.join(working_directory,"Input Data","Streamflow","Q_Lookout_USGS_1950_2026.csv")
outputfolder= os.path.join(working_directory,"Intermediate_Outputs")

# watershed areas convert ha to square m (taken from HJA website)
lookoutarea=6242*10000
mackarea=580*10000
ws01area=96*10000
ws02area=60*10000
ws03area=101*10000
ws08area=21.4*10000
ws09area=8.5*10000
ws10area=10.2*10000

In [14]:
#read in longterm daily summaries
olddata= pd.read_csv(olddatafile, usecols=[2,3,4,5,8,10])
olddata["SITECODE"]=olddata["SITECODE"].astype('string')
olddata["WATERYEAR"]=pd.to_numeric(olddata["WATERYEAR"],errors='coerce',downcast='integer') 
olddata["DATE"]=pd.to_datetime(olddata["DATE"],errors='coerce')
olddata["MEAN_Q"]=pd.to_numeric(olddata["MEAN_Q"],errors='coerce') 
olddata["MEAN_Q_AREA"]=pd.to_numeric(olddata["MEAN_Q_AREA"],errors='coerce')  
#filter out unnecessary watersheds
olddata=olddata[olddata["SITECODE"]!="GSWS06"]
olddata=olddata[olddata["SITECODE"]!="GSWS07"]
olddata=olddata[olddata["SITECODE"]!="GSWSMF"]

#convert MEAN_Q_AREA to metric
# conversion from cubic feet per second per square mile to mm per second is 1.09331956 × 10^-5 mm per second
olddata["MEAN_Q_AREA_mms"]=olddata["MEAN_Q_AREA"]*1.09331956*10**(-5)
# print(olddata)
#set MEAN_Q_AREA to NaN if ESTCODE is not A or E
olddata.loc[~olddata['ESTCODE'].isin(["A","E"]), 'MEAN_Q_AREA_mms'] = np.nan
# print(olddata[olddata['MEAN_Q_AREA_mms'].isna()])
olddata_pivot=olddata.pivot_table(values='MEAN_Q_AREA_mms',index="DATE", columns="SITECODE").reset_index()
#fill in GSWSMA for GSWSMC when GSWSMC is blank
olddata_pivot['GSWSMC'] = np.where(olddata_pivot['GSWSMC'].isna(), olddata_pivot['GSWSMA'], olddata_pivot['GSWSMC'])
olddata_pivot=olddata_pivot.drop(columns=['GSWSMA'])
print(olddata_pivot)

SITECODE       DATE    GSLOOK        GSWS01    GSWS02    GSWS03        GSWS08  \
0        1949-10-01  0.000005           NaN       NaN       NaN           NaN   
1        1949-10-02  0.000005           NaN       NaN       NaN           NaN   
2        1949-10-03  0.000005           NaN       NaN       NaN           NaN   
3        1949-10-04  0.000006           NaN       NaN       NaN           NaN   
4        1949-10-05  0.000025           NaN       NaN       NaN           NaN   
...             ...       ...           ...       ...       ...           ...   
27389    2024-09-26  0.000005  1.104253e-06  0.000004  0.000002  6.450585e-07   
27390    2024-09-27  0.000005  1.016787e-06  0.000004  0.000002  5.138602e-07   
27391    2024-09-28  0.000005  9.730544e-07  0.000003  0.000002  4.482610e-07   
27392    2024-09-29  0.000005  9.839876e-07  0.000003  0.000002  3.935950e-07   
27393    2024-09-30  0.000005  9.839876e-07  0.000003  0.000002  3.717287e-07   

SITECODE    GSWS09        G

In [15]:
#read in new data
newdata=pd.read_csv(newdatafile, skiprows=5, usecols=[2,3,4,5,6,7,8,9,10,15,16,17,18,19,20], sep = ",", quotechar='"',
                    names=["Datetime", 
                           "GSWSMC", 
                           "Flag_GSMACK_DISCHARGE", 
                           "GSWS01", 
                           "Flag_GSWS01_DISCHARGE",
                           "GSWS02",
                           "Flag_GSWS02_DISCHARGE",
                           "GSWS03",
                           "Flag_GSWS03_DISCHARGE",
                           "GSWS08",
                           "Flag_GSWS08_DISCHARGE",
                           "GSWS09",
                           "Flag_GSWS09_DISCHARGE",
                           "GSWS10",
                           "Flag_GSWS10_DISCHARGE"])
newdata["Datetime"]=pd.to_datetime(newdata["Datetime"],errors='coerce')

# create date column
newdata['date']=newdata["Datetime"].dt.date

#convert discharge from cfs to cms by multiplying by 0.0283168
newdata["GSWSMC_cms"]=newdata["GSWSMC"]*0.0283168
newdata["GSWS01_cms"]=newdata["GSWS01"]*0.0283168
newdata["GSWS02_cms"]=newdata["GSWS02"]*0.0283168
newdata["GSWS03_cms"]=newdata["GSWS03"]*0.0283168
newdata["GSWS08_cms"]=newdata["GSWS08"]*0.0283168
newdata["GSWS09_cms"]=newdata["GSWS09"]*0.0283168
newdata["GSWS10_cms"]=newdata["GSWS10"]*0.0283168

#convert to unit discharge in mm/s by dividing by area in square m and multiplying by 1000
newdata["GSWSMC_mms"]=newdata["GSWSMC_cms"]/mackarea*1000
newdata["GSWS01_mms"]=newdata["GSWS01_cms"]/ws01area*1000
newdata["GSWS02_mms"]=newdata["GSWS02_cms"]/ws02area*1000
newdata["GSWS03_mms"]=newdata["GSWS03_cms"]/ws03area*1000
newdata["GSWS08_mms"]=newdata["GSWS08_cms"]/ws08area*1000
newdata["GSWS09_mms"]=newdata["GSWS09_cms"]/ws09area*1000
newdata["GSWS10_mms"]=newdata["GSWS10_cms"]/ws10area*1000

#set discharge_mms to NaN if Flag is not blank or E
newdata.loc[newdata['Flag_GSMACK_DISCHARGE'].isna(), 'Flag_GSMACK_DISCHARGE'] = "A"
newdata.loc[~newdata['Flag_GSMACK_DISCHARGE'].isin(["A","E"]), 'GSWSMC_mms'] = np.nan
newdata.loc[newdata['Flag_GSWS01_DISCHARGE'].isna(), 'Flag_GSWS01_DISCHARGE'] = "A"
newdata.loc[~newdata['Flag_GSWS01_DISCHARGE'].isin(["A","E"]), 'GSWS01_mms'] = np.nan
newdata.loc[newdata['Flag_GSWS02_DISCHARGE'].isna(), 'Flag_GSWS02_DISCHARGE'] = "A"
newdata.loc[~newdata['Flag_GSWS02_DISCHARGE'].isin(["A","E"]), 'GSWS02_mms'] = np.nan
newdata.loc[newdata['Flag_GSWS03_DISCHARGE'].isna(), 'Flag_GSWS03_DISCHARGE'] = "A"
newdata.loc[~newdata['Flag_GSWS03_DISCHARGE'].isin(["A","E"]), 'GSWS03_mms'] = np.nan
newdata.loc[newdata['Flag_GSWS08_DISCHARGE'].isna(), 'Flag_GSWS08_DISCHARGE'] = "A"
newdata.loc[~newdata['Flag_GSWS08_DISCHARGE'].isin(["A","E"]), 'GSWS08_mms'] = np.nan
newdata.loc[newdata['Flag_GSWS09_DISCHARGE'].isna(), 'Flag_GSWS09_DISCHARGE'] = "A"
newdata.loc[~newdata['Flag_GSWS09_DISCHARGE'].isin(["A","E"]), 'GSWS09_mms'] = np.nan
newdata.loc[newdata['Flag_GSWS10_DISCHARGE'].isna(), 'Flag_GSWS10_DISCHARGE'] = "A"
newdata.loc[~newdata['Flag_GSWS10_DISCHARGE'].isin(["A","E"]), 'GSWS10_mms'] = np.nan

C:\Users\duffshan\AppData\Local\Temp\ipykernel_28604\2535442615.py:2: DtypeWarning: Columns (6,8,10,16,18,20) have mixed types. Specify dtype option on import or set low_memory=False.
  newdata=pd.read_csv(newdatafile, skiprows=5, usecols=[2,3,4,5,6,7,8,9,10,15,16,17,18,19,20], sep = ",", quotechar='"',
C:\Users\duffshan\AppData\Local\Temp\ipykernel_28604\2535442615.py:42: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'A' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  newdata.loc[newdata['Flag_GSMACK_DISCHARGE'].isna(), 'Flag_GSMACK_DISCHARGE'] = "A"


In [16]:
#group by date and find mean Q
newdata_group_date_meanQ=newdata.groupby("date").agg(daily_meanQ_GSWSMC=("GSWSMC_mms","mean"),
                                                     daily_meanQ_GSWS01=("GSWS01_mms","mean"),
                                                     daily_meanQ_GSWS02=("GSWS02_mms","mean"),
                                                     daily_meanQ_GSWS03=("GSWS03_mms","mean"),
                                                     daily_meanQ_GSWS08=("GSWS08_mms","mean"),
                                                     daily_meanQ_GSWS09=("GSWS09_mms","mean"),
                                                     daily_meanQ_GSWS10=("GSWS10_mms","mean")).reset_index()

# create month and year columns
newdata_group_date_meanQ['DATE']=pd.to_datetime(newdata_group_date_meanQ["date"])

# trim data to max date of old data
old_max_date = olddata['DATE'].max()
newdata_group_date_meanQ=newdata_group_date_meanQ[newdata_group_date_meanQ["DATE"]>pd.to_datetime(old_max_date)]
print(newdata_group_date_meanQ)

            date  daily_meanQ_GSWSMC  daily_meanQ_GSWS01  daily_meanQ_GSWS02  \
2685  2024-10-01            0.000006        9.190260e-07            0.000003   
2686  2024-10-02            0.000006        8.937490e-07            0.000003   
2687  2024-10-03            0.000006        8.937490e-07            0.000003   
2688  2024-10-04            0.000006        9.264411e-07            0.000003   
2689  2024-10-05            0.000006        1.018209e-06            0.000003   
...          ...                 ...                 ...                 ...   
3046  2025-09-27            0.000005        7.433160e-07            0.000003   
3047  2025-09-28            0.000005        7.433160e-07            0.000003   
3048  2025-09-29            0.000005        7.620587e-07            0.000003   
3049  2025-09-30            0.000005        9.711778e-07            0.000003   
3050  2025-10-01            0.000008        1.058930e-06            0.000003   

      daily_meanQ_GSWS03  daily_meanQ_G

In [17]:
#rename columns in new data
new_data_rename=newdata_group_date_meanQ.rename(columns={"daily_meanQ_GSWSMC":"GSWSMC","daily_meanQ_GSWS01":"GSWS01","daily_meanQ_GSWS02":"GSWS02","daily_meanQ_GSWS03":"GSWS03","daily_meanQ_GSWS08":"GSWS08","daily_meanQ_GSWS09":"GSWS09","daily_meanQ_GSWS10":"GSWS10"})
#merge old and new data
old_plus_new=pd.concat([olddata_pivot,new_data_rename],axis=0).reset_index()
print(old_plus_new)

       index       DATE    GSLOOK        GSWS01    GSWS02    GSWS03  \
0          0 1949-10-01  0.000005           NaN       NaN       NaN   
1          1 1949-10-02  0.000005           NaN       NaN       NaN   
2          2 1949-10-03  0.000005           NaN       NaN       NaN   
3          3 1949-10-04  0.000006           NaN       NaN       NaN   
4          4 1949-10-05  0.000025           NaN       NaN       NaN   
...      ...        ...       ...           ...       ...       ...   
27755   3046 2025-09-27       NaN  7.433160e-07  0.000003  0.000002   
27756   3047 2025-09-28       NaN  7.433160e-07  0.000003  0.000002   
27757   3048 2025-09-29       NaN  7.620587e-07  0.000003  0.000002   
27758   3049 2025-09-30       NaN  9.711778e-07  0.000003  0.000002   
27759   3050 2025-10-01       NaN  1.058930e-06  0.000003  0.000003   

             GSWS08        GSWS09        GSWS10    GSWSMC        date  
0               NaN           NaN           NaN       NaN         NaN  
1  

In [18]:
# read in Lookout new data
lookout_new_data = pd.read_csv(lookout_new_data_file, skiprows=1,usecols=[7,8], names=["Datetime","Q_cfs"])
lookout_new_data["Datetime"]=pd.to_datetime(lookout_new_data["Datetime"],errors='coerce')
#convert discharge from cfs to cms by multiplying by 0.0283168
lookout_new_data["Q_cms"]=lookout_new_data["Q_cfs"]*0.0283168
#convert to unit discharge in mm/s by dividing by area in square m and multiplying by 1000
lookout_new_data["Q_mms"]=lookout_new_data["Q_cms"]/lookoutarea*1000
# create date, month, and year columns
lookout_new_data['DATE']=lookout_new_data["Datetime"].dt.date
lookout_new_data["DATE"]=pd.to_datetime(lookout_new_data["DATE"],errors='coerce')
# trim data in olddata (>2020)
lookout_new_data=lookout_new_data[lookout_new_data["DATE"]>pd.to_datetime(old_max_date)]
print(lookout_new_data)

      Datetime  Q_cfs     Q_cms     Q_mms       DATE
0   2026-06-01   28.6  0.809860  0.000013 2026-06-01
1   2026-05-31   31.2  0.883484  0.000014 2026-05-31
2   2026-05-30   36.8  1.042058  0.000017 2026-05-30
3   2026-05-29   44.4  1.257266  0.000020 2026-05-29
4   2026-05-28   25.8  0.730573  0.000012 2026-05-28
..         ...    ...       ...       ...        ...
604 2024-10-05   10.9  0.308653  0.000005 2024-10-05
605 2024-10-04   10.6  0.300158  0.000005 2024-10-04
606 2024-10-03   10.4  0.294495  0.000005 2024-10-03
607 2024-10-02   10.4  0.294495  0.000005 2024-10-02
608 2024-10-01   10.4  0.294495  0.000005 2024-10-01

[609 rows x 5 columns]


In [19]:
#merge old_plus_new with lookout data
#rename columns in lookout data
lookout_rename=lookout_new_data.rename(columns={"Q_mms":"GSLOOK"})
print(lookout_rename)
#append newdata to olddata
totalrecord = pd.merge(old_plus_new,lookout_rename,how="left", on= "DATE",suffixes=("",'_y'))
print(totalrecord)
totalrecord['GSLOOK'] = np.where(totalrecord['GSLOOK'].isna(), totalrecord['GSLOOK_y'], totalrecord['GSLOOK'])
print(totalrecord)
totalrecord=totalrecord.drop(columns=['GSLOOK_y','Q_cfs','Datetime',"Q_cms","index"])
print(totalrecord)

      Datetime  Q_cfs     Q_cms    GSLOOK       DATE
0   2026-06-01   28.6  0.809860  0.000013 2026-06-01
1   2026-05-31   31.2  0.883484  0.000014 2026-05-31
2   2026-05-30   36.8  1.042058  0.000017 2026-05-30
3   2026-05-29   44.4  1.257266  0.000020 2026-05-29
4   2026-05-28   25.8  0.730573  0.000012 2026-05-28
..         ...    ...       ...       ...        ...
604 2024-10-05   10.9  0.308653  0.000005 2024-10-05
605 2024-10-04   10.6  0.300158  0.000005 2024-10-04
606 2024-10-03   10.4  0.294495  0.000005 2024-10-03
607 2024-10-02   10.4  0.294495  0.000005 2024-10-02
608 2024-10-01   10.4  0.294495  0.000005 2024-10-01

[609 rows x 5 columns]
       index       DATE    GSLOOK        GSWS01    GSWS02    GSWS03  \
0          0 1949-10-01  0.000005           NaN       NaN       NaN   
1          1 1949-10-02  0.000005           NaN       NaN       NaN   
2          2 1949-10-03  0.000005           NaN       NaN       NaN   
3          3 1949-10-04  0.000006           NaN       Na

In [ ]:
# create month and year columns
totalrecord['year']=totalrecord["DATE"].dt.year
totalrecord['month']=totalrecord["DATE"].dt.month

#create wateryear column
def getWateryear(row):
       if row["month"]<10:
              wateryear=row["year"]
       else:
              wateryear=row["year"]+1
       return wateryear
totalrecord["WATERYEAR"]=totalrecord.apply(getWateryear, axis=1)
totalrecord=totalrecord[totalrecord['WATERYEAR']<2026]
print(totalrecord)

            DATE    GSLOOK        GSWS01    GSWS02    GSWS03        GSWS08  \
0     1949-10-01  0.000005           NaN       NaN       NaN           NaN   
1     1949-10-02  0.000005           NaN       NaN       NaN           NaN   
2     1949-10-03  0.000005           NaN       NaN       NaN           NaN   
3     1949-10-04  0.000006           NaN       NaN       NaN           NaN   
4     1949-10-05  0.000025           NaN       NaN       NaN           NaN   
...          ...       ...           ...       ...       ...           ...   
27754 2025-09-26  0.000005  7.433160e-07  0.000003  0.000002  1.455536e-07   
27755 2025-09-27  0.000005  7.433160e-07  0.000003  0.000002  1.455536e-07   
27756 2025-09-28  0.000005  7.433160e-07  0.000003  0.000002  1.455536e-07   
27757 2025-09-29  0.000005  7.620587e-07  0.000003  0.000002  1.830907e-07   
27758 2025-09-30  0.000005  9.711778e-07  0.000003  0.000002  3.170202e-07   

             GSWS09        GSWS10    GSWSMC        date  year  

In [23]:
#export to a csv
output_path = os.path.join(outputfolder, "Daily_meanQ_allsites_1950_2025.csv")
totalrecord.to_csv(output_path, index=False)